###Usecase1: Storage & Security
**Databricks integration with ADLS & BLOB implementing different types of Security protocols (IAM/RBAC/ACLs, SAS, Accesskeys, SP, Managed Identity )**

![](azure-storage-security.png)

####1. A *Shared Access Signature (SAS)* token is a secure URI string that grants restricted, temporary access rights to Microsoft Azure Storage resources - (Fast, Resource (Storage Account/Container) Level, Temporary, Vendors/3rd party users)

In [0]:
#Databricks Read data from Az BLOB using SAS token
#https://izblobmahe.blob.core.windows.net/container1/sourcedir/custs_header
storage_account = "aparnablobstorageacc"
container_name = "blobcontainer1"
sas_token = "replacewithyourSASToken"
spark.conf.set(f"fs.azure.sas.{container_name}.{storage_account}.blob.core.windows.net", sas_token)
path = f"wasbs://{container_name}@{storage_account}.blob.core.windows.net/sourcedir/"
df = spark.read.csv(path, header=True)
display(df)

In [0]:
adlsacct='wd36adlsstoragedata'
containername='adlscontainer'
accesskey='replacewithproperaccesskey'
spark.conf.set(f"fs.azure.account.key.{adlsacct}.dfs.core.windows.net", accesskey)
path = f"abfss://{containername}@{adlsacct}.dfs.core.windows.net/inceptezbronzedir/"
#df.write.mode("overwrite").csv(path, header=True)
df.write.mode("overwrite").json(path)

In [0]:
adlsacct='wd36adlsstoragedata'
containername='adlscontainer'
accesskey='replacewithproperaccesskey'
spark.conf.set(f"fs.azure.account.key.{adlsacct}.dfs.core.windows.net", accesskey)
path = f"abfss://{containername}@{adlsacct}.dfs.core.windows.net/inceptezbronzedir/"
#df.write.mode("overwrite").csv(path, header=True)
df.write.mode("overwrite").json(path)

**2. An Azure Storage Access Key is a 512-bit, auto-generated key that acts as a root password for your Azure storage account. It grants full, unrestricted access to the storage account configuration and all its data**

In [0]:
adlsacct='wd36adlsstoragedata'
containername='adlscontainer'
accesskey='replacewithproperaccesskey'
spark.conf.set(f"fs.azure.account.key.{adlsacct}.dfs.core.windows.net", accesskey)
path = f"abfss://{containername}@{adlsacct}.dfs.core.windows.net/inceptezbronzedir/"
#df.show(10)
df.write.mode("overwrite").csv(path, header=True)
#df.write.mode("overwrite").json(path)

In [0]:
#Just for understanding, we can write the data also back to the blob storage.
df2=df.where("age>60")
storage_account = "selvablobstorageacc"
container_name = "blobcontainer1"
sas_token = "replacewithproperaccesskey"
spark.conf.set(f"fs.azure.sas.{container_name}.{storage_account}.blob.core.windows.net",sas_token)
path = f"wasbs://{container_name}@{storage_account}.blob.core.windows.net/targetdir/"
df2.write.mode("overwrite").csv(path, header=True)

**Very Important for Interview**
**How to configure Azure Keyvault to store Account key or any other passwords securely and access it using Databricks secret scope:**

**What is Azure Keyvault? It is the centralized service for managing, storing, and securing sensitive information—such as credentials, API keys, and connection strings—that are essential for building secure, automated data pipelines**

**And 'How we can use Azure Keyvault with Databricks Secret' to secure our access keys/passwords with highest level of security**

Step 1: Store the Secret in Azure Key Vault via Azure CLI
You will need the Azure CLI (az) installed and authenticated (az login) for these commands.

1. Create an Azure Key Vault:
az account set --subscription "e68a829e-83e8-42bc-8494-b4ceb80be6c7"
az provider register --namespace Microsoft.KeyVault
az provider show --namespace Microsoft.KeyVault --query "registrationState"

az keyvault create \
  --name "izkeyvaultwe47" \
  --resource-group "wd36-rg1" \
  --location "eastus"

Capture the ID Assignee ID:
"id": "/subscriptions/e68a829e-83e8-42bc-8494-b4ceb80be6c7/resourceGroups/wd36-rg1/providers/Microsoft.KeyVault/vaults/izkeyvaultwe4"

2. Store your access key as a secret in the Key Vault:

az role assignment create \
  --role "Key Vault Secrets Officer" \
  --assignee "232d0fbc-d8bd-412e-b7bf-16a5ec95ae93" \
  --scope "/subscriptions/e68a829e-83e8-42bc-8494-b4ceb80be6c7/resourcegroups/wd36-rg1/providers/microsoft.keyvault/vaults/izkeyvaultwe4"

az keyvault secret set \
  --vault-name "izkeyvaultwe47" \
  --name "adls-access-key-we471" \
  --value "giveyouraccesskey"

az keyvault secret set \
  --vault-name "izkeyvaultwe47" \
  --name "adls-sp-secret-we47" \
  --value "giveyoursecretvalue"
  
3. Get the Key Vault Resource ID and DNS Name (You will need these for Databricks):

az keyvault show --name "izkeyvaultwe47" --query "id" -o tsv

\#Resource ID Output example: /subscriptions/e68a829e-83e8-42bc-8494-b4ce6c7/resourceGroups/wd36-rg1/providers/Microsoft.KeyVault/vaults/izkeyvaultwe47

az keyvault show --name "izkeyvaultwe47" --query "properties.vaultUri" -o tsv

\# Output example: https://izkeyvaultwe47.vault.azure.net/


Step 2: Link Key Vault (using uri generated in the above step) to Databricks (One-time setup)
Note: The Databricks CLI currently only supports creating Databricks-native secret scopes. To create an Azure Key Vault-backed scope, you must use the Databricks web interface.

Navigate to https://<your-databricks-workspace-url>#secrets/createScope

Eg:
https://adb-7405605290162515.15.azuredatabricks.net/#secrets/createScope

Enter a Scope Name (e.g., we47scope2).

Enter the DNS Name and Resource ID you retrieved in the previous step:
#Resource ID Output example: /subscriptions/e68a829e-83e8-42bc-8494-bb80be6c7/resourceGroups/wd36-rg1/providers/Microsoft.KeyVault/vaults/izkeyvaultwe47
Click Create.

Step 3: Assign role as a key vault secret user

az role assignment create \
  --role "Key Vault Secrets User" \
  --assignee-object-id "530cd74e-90c7-4365-8fed-2d91d79fb006" \
  --assignee-principal-type "ServicePrincipal" \
  --scope "/subscriptions/e68a829e-83e8-42bc-8494-b4be6c7/resourceGroups/wd36-rg1/providers/Microsoft.KeyVault/vaults/izkeyvaultwe47"

In [0]:
#Step 3: Refactored PySpark Code for Databricks
# Define storage account and container variables
adlsacct = 'wd36adlsstoragedata'
containername = 'adlscontainer'

# Securely fetch the access key from the Azure Key Vault-backed secret scope
# Syntax: dbutils.secrets.get(scope="<scope-name>", key="<secret-name>")
accesskey = dbutils.secrets.get(scope="we47scope2",key="adls-access-key-we471")
#secret is not coming from databricks secret scope, rather it is coming from azure keyvault
print(accesskey)
# Set the Spark configuration using the retrieved key
spark.conf.set(f"fs.azure.account.key.{adlsacct}.dfs.core.windows.net", accesskey)

# Define the target ADLS Gen2 path
path = f"abfss://{containername}@{adlsacct}.dfs.core.windows.net/inceptezbronzedir/"

display(spark.read.csv(path))

**3. In Microsoft Azure, a Service Principal is a non-human identity created specifically for applications, hosted services, and automated tools to access Azure resources - Note: Comparing with SAS or Access keys, preferably use Service Principal with IAM & RBAC**

In [0]:
#This will be done by the Platform Engineers, not by Dataengineers
#Platform engineers will create Service principal & provide service_principal_id & service_principal_secret to the users for using inside their programs (CLI/Python/Pyspark)
#Search for Microsoft Entra & copy the Tenant Id
tenant_id = "089cf941-fd292-985784dc42d6"
#Microsoft Entra ID -> Manage -> App Registration -> All Applications -> Click on the service principal (created already) ->  Copy Application (client) ID or Overview -> Application (client) ID (copy) 
service_principal_id = "530cd74e-90c78fed-2d91d79fb006"
#Microsoft Entra ID -> Manage -> App Registration -> All Applications -> Click on the service principal (created already) ->  Manage -> Certificates & Secrets -> Client Secrets -> New client secret -> Value (copy)
service_principal_secret ="replacewithproperaccesskey"

#Admin will give RBAC in IAM
#Stroage Account -> Access Control (IAM) -> Add role assignment -> Storage Blob Data Contributor -> Select the service principal (created already)

#storage_account = "wd36adlsstorageacct2"
#container_name = "containerwd362"
storage_account='wd36adlsstoragedata'
container_name='adlscontainer'

# Standard Spark configurations for Service Principal
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")#open authorization
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", service_principal_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", service_principal_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

bronzepath = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/inceptezbronzedir/"
df = spark.read.csv(bronzepath,header=True)
#display(df)
silverpath = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/silvertargetdir/"
df.where("age>60").write.mode("overwrite").format("delta").save(silverpath)

#df = spark.read.csv(silverpath,header=True)
#display(df)
#df.write.mode("overwrite").format("delta").save('/Volumes/wd36_workspace1/default/volume1/custtargetdata')

**4. A Managed Identity is a feature in Microsoft Entra ID (formerly Azure AD) that provides an automatically managed identity for applications and services running on Azure. It allows these applications to authenticate securely with other Azure resources without developers needing to manually manage, rotate, or store credentials (like passwords, connection strings, or access keys) in their code.**
Comparing with SAS or Access keys or Service Principal use Managed Identity, which is the best option

**Enable Managed Identity for Databricks Serverless(preferably works using Managed Identity) - Done by Platform Team not by Developers**

Step 1: Azure Side - Create the Azure Access Connector (The Access Connector is the "container" for the Managed Identity that Databricks will use.)
1. In the Azure Portal, search for Access Connector for Azure Databricks.
2. Click Create.
3. Assign it a name (e.g., serverless-identity-connector) and a Resource Group.
4. Once created, go to the access connector Overview tab and copy the Resource ID 
/subscriptions/e68a829e-83e8-42bc-8494-b4ceb80be6c7/resourceGroups/wd36-rg1/providers/Microsoft.Databricks/accessConnectors/db-azure-managed-identity-we47

Step 2: Azure IAM/RBAC Side - Grant Permissions in Azure Storage (Now, tell Azure that this new identity has permission to read your data.)

1. Go to your Azure Storage Account -> Access Control (IAM).
2. Click Add role assignment (RBAC)
3. Select Storage Blob Data Contributor.
4. Under Assign access to, select Managed Identity.
5. Choose your "Access Connector for azure databricks" from the list and save.
wd36blobstorageacct1

Step 3: Databricks Side - Create a Storage Credential in Databricks (This step tells your Databricks Workspace to use the Azure identity just we created)

1. In Databricks, go to Catalog > Click Setting Gear icon in the top > External Locations
2. Click External Locations > Credentials.
3. Click Create credential > Storage Credential
4. Credential name: give some name (inceptez_managed_id)
5. Access Connector ID: Paste the Resource ID you copied in Step 1.4
6. Click Create.

Step 4: Databricks Unity Catalog Side (to route my data access from azure storage using unity catalog external location) - Define an External Location (An External Location links a specific storage path to your credential.)
1. In Catalog Explorer (settings button) go to External Data > External Locations.
2. Click Create location.
3. URL: Enter container path (e.g., abfss://my-container@my-account.dfs.core.windows.net/).
abfss://wd36container1@wd36blobstorageacct1.dfs.core.windows.net/
4. Storage Credential: Select the one created in Step 3.
5. Click Create.

In [0]:

display(spark.read.csv("abfss://container11@wd36blobstorageacct1.dfs.core.windows.net/sampledir/",header=True))